In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Read data from the file you just uploaded
df = pd.read_csv('hand_gesture_data.csv')
X = df.iloc[:, :-1].values # Get 63 coordinate columns
y = df.iloc[:, -1].values  # Get the label column (0, 1, 2)

# 2. IMPORTANT PREPROCESSING FUNCTION (Normalization)
def preprocess_landmarks(X_data):
    processed_data = []
    for row in X_data:
        # Convert to matrix (21, 3) for easy processing
        landmarks = row.reshape(21, 3)
        # Take the wrist (Landmark 0) as the origin
        wrist = landmarks[0]
        relative_landmarks = landmarks - wrist
        # Flatten back to 63 elements
        processed_data.append(relative_landmarks.flatten())
    return np.array(processed_data)

X_processed = preprocess_landmarks(X)

# 3. Split the dataset: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"Data is ready! Training set: {len(X_train)} samples.")

Dữ liệu đã sẵn sàng! Tập Train: 381 mẫu.


In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(63,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.2), # Help the model not to overfit
    tf.keras.layers.Dense(3, activation='softmax') # 3 outputs for 3 labels
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary() # View the layer-by-layer structure of the AI

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,627 (10.26 KB)

 Trainable params: 2,627 (10.26 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=100,      # Learn again and again 100 times
    batch_size=32,   # Each time learn 32 samples
    validation_data=(X_test, y_test)
)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.3937 - loss: 1.0683 - val_accuracy: 0.4479 - val_loss: 1.0482
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4672 - loss: 1.0373 - val_accuracy: 0.5521 - val_loss: 1.0142
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5039 - loss: 1.0107 - val_accuracy: 0.5729 - val_loss: 0.9843
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4882 - loss: 0.9828 - val_accuracy: 0.5625 - val_loss: 0.9537
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5407 - loss: 0.9558 - val_accuracy: 0.5833 - val_loss: 0.9199
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5669 - loss: 0.9223 - val_accuracy: 0.6354 - val_loss: 0.8845
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6483 - loss: 0.8813 - val_accuracy: 0.6667 - val_loss: 0.8496
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6430 - loss: 0.8557 - val_accuracy: 0.7188 - 

In [ ]:
# Convert to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save file
with open('gesture_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

print("Successfully created gesture_classifier.tflite file!")

Saved artifact at '/tmp/tmpd9o_xf7y'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 63), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134923985850576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134923985851536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134923985849616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134923985851152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134923985853072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134923985854032: TensorSpec(shape=(), dtype=tf.resource, name=None)
Đã tạo file gesture_classifier.tflite thành công!
